# JASSIM AI — Google Colab GPU Serverدفتر تشغيل تجريبي لخادم JASSIM AI. يحفظ SQLite في Google Drive، ويشغّل Ollama والنموذج، ثم يفتح نفق HTTPS مؤقتاً عبر Cloudflare.**ملاحظات:** جلسات Colab المجانية مؤقتة وGPU غير مضمون. توجد خلية Watchdog لإعادة تشغيل العمليات المتعطلة، لكنها لا تمنع إيقاف جلسة Colab. لا تشارك رابط النفق أو رمز API مع الآخرين. لا تضع أسراراً في GitHub.من قائمة Runtime اختر Change runtime type ثم GPU إن كان متاحاً.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')from pathlib import Pathimport os, subprocess, time, json, secrets, getpass, requests, re, textwrapPROJECT = Path('/content/JASSIM-AI-Android')DATA_DIR = Path('/content/drive/MyDrive/JASSIM_AI/data')DATA_DIR.mkdir(parents=True, exist_ok=True)print('Drive mounted. Persistent database directory:', DATA_DIR)

## 1. تنزيل ملفات خادم JASSIMنستخدم نسخة المستودع الحالية دون تعديل ملفات التطبيق أو الذاكرة الموجودة فيه.

In [ ]:
if not PROJECT.exists():    subprocess.run(['git','clone','--depth','1','https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git',str(PROJECT)], check=True)SERVER = PROJECT / 'backend' / 'server.py'assert SERVER.exists(), 'backend/server.py is missing'print('Server source ready:', SERVER)

## 2. إعداد رمز الحمايةأنشئ رمزاً عشوائياً أو أدخل رمزاً تملكه. سيُستخدم في التطبيق والموقع. لا ترسله في المحادثة ولا تحفظه في GitHub.

In [ ]:
API_TOKEN = secrets.token_urlsafe(48)print('Generated a private API token for this Colab session.')print('Keep this notebook session private. Copy the token from the next cell output only if you need to configure the Android app/dashboard.')print(API_TOKEN)os.environ['JASSIM_API_TOKEN'] = API_TOKENos.environ['JASSIM_CORS_ORIGIN'] = 'https://stevennpllk-crypto.github.io'os.environ['JASSIM_HOST'] = '127.0.0.1'os.environ['JASSIM_PORT'] = '8787'os.environ['JASSIM_DATA_DIR'] = str(DATA_DIR)os.environ['JASSIM_OLLAMA_URL'] = 'http://127.0.0.1:11434'os.environ['JASSIM_OLLAMA_MODEL'] = 'qwen3.5:4b'os.environ['JASSIM_OLLAMA_KEEP_ALIVE'] = '-1'os.environ['JASSIM_NUM_CTX'] = '8192'os.environ['JASSIM_NUM_PREDICT'] = '2048'os.environ['JASSIM_HISTORY_CHAR_BUDGET'] = '24000'os.environ['JASSIM_REQUEST_TIMEOUT'] = '240'

## 3. تثبيت وتشغيل Ollamaنستخدم Qwen 2.5 3B كخط أساس أقوى من 1.5B. يحتاج ذاكرة أكبر ووقت تنزيل أطول؛ اختر GPU من إعدادات Colab إن توفر. يبقى النموذج محمّلاً طوال بقاء عملية Ollama، لكن Colab قد ينهي الجلسة تلقائياً.

In [ ]:
install = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)print(install.stdout[-3000:])if install.returncode != 0:    print(install.stderr[-3000:])    raise RuntimeError('Ollama installation failed')ollama_log = open('/content/ollama.log','w')ollama_proc = subprocess.Popen(['ollama','serve'], stdout=ollama_log, stderr=subprocess.STDOUT)for _ in range(30):    try:        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)        break    except Exception:        time.sleep(2)else:    raise RuntimeError('Ollama did not start. Check /content/ollama.log')pull = subprocess.run(['ollama','pull','qwen3.5:4b'], text=True)if pull.returncode != 0:    raise RuntimeError('Model download failed')print('Ollama and qwen3.5:4b are ready. Keep-alive is enabled for this runtime.')

## 4. تشغيل JASSIM API مع ذاكرة Driveتُحفظ قاعدة البيانات داخل مجلد JASSIM_AI في Drive. لا تحذف هذا المجلد إذا أردت الاحتفاظ بالمحادثات.

In [ ]:
api_log = open('/content/jassim-api.log','w')api_proc = subprocess.Popen(['python',str(SERVER)], cwd=str(PROJECT/'backend'), env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT)for _ in range(30):    try:        health = requests.get('http://127.0.0.1:8787/health', timeout=2).json()        if health.get('ok'): break    except Exception:        time.sleep(2)else:    raise RuntimeError('JASSIM API did not start. Check /content/jassim-api.log')print(json.dumps(health, ensure_ascii=False, indent=2))

## 5. اختبار المحرك والحمايةهذا الاختبار يرسل رسالة تجريبية إلى النموذج ويحفظها في سجل المحادثة.

In [ ]:
headers = {'Authorization': 'Bearer ' + API_TOKEN, 'Content-Type':'application/json'}test = requests.post('http://127.0.0.1:8787/api/chat', headers=headers, json={'message':'عرّف بنفسك في جملة قصيرة بالعربية.','conversation_id':'colab-check'}, timeout=180)print('HTTP:', test.status_code)print(json.dumps(test.json(), ensure_ascii=False, indent=2))assert test.ok, 'API chat test failed'

## 6. فتح رابط HTTPS مؤقتسنشغّل Cloudflare Quick Tunnel. الرابط يتغير عند إعادة تشغيل الجلسة، وهو ليس استضافة دائمة.

In [ ]:
deb = '/content/cloudflared.deb'subprocess.run(['bash','-lc',f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {deb} && dpkg -i {deb}"], check=True)cf_log = open('/content/cloudflared.log','w')cf_proc = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8787'], stdout=cf_log, stderr=subprocess.STDOUT, text=True)tunnel_url = Nonefor _ in range(60):    time.sleep(2)    try:        log_text = Path('/content/cloudflared.log').read_text(errors='ignore')        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)        if match:            tunnel_url = match.group(0)            break    except Exception:        passif not tunnel_url:    print(Path('/content/cloudflared.log').read_text(errors='ignore')[-4000:])    raise RuntimeError('Could not obtain a Cloudflare tunnel URL')print('JASSIM API HTTPS URL:', tunnel_url)print('Health:', requests.get(tunnel_url + '/health', timeout=20).json())

## 7. ربط التطبيق والموقعفي إعدادات تطبيق Android ولوحة التحكم:- Server URL: انسخ رابط HTTPS الذي ظهر في الخلية السابقة.- API Token: انسخ الرمز الخاص الذي وُلّد في خلية الحماية.احتفظ بالرابط والرمز بشكل خاص. لا تنشرهما ولا ترسلهما إلى أشخاص آخرين. عند انتهاء جلسة Colab أعد تشغيل الخلايا؛ قد تحصل على رابط جديد، لكن بيانات SQLite تبقى في Drive.**مهم:** لا تترك الخلية الأخيرة دون مراقبة؛ إيقاف جلسة Colab يوقف النموذج والخادم والنفق.

In [ ]:
print('API URL:', tunnel_url)print('API Token:', API_TOKEN)print('Persistent DB folder:', DATA_DIR)print('Keep this Colab runtime active while using JASSIM.')

## 8. مراقبة وإعادة تشغيل تلقائية (Watchdog)شغّل الخلية التالية مرة واحدة. تحاول إعادة تشغيل Ollama وAPI والنفق إذا تعطلت العمليات أو تكرر فشل الفحص. عند تغير رابط النفق اطبعه مجدداً وحدّث إعدادات التطبيق. هذه الحماية تعمل فقط ما دامت جلسة Colab نفسها حية؛ لا تمنع فصل Colab.

In [ ]:
# Watchdog: restarts failed services while this Colab runtime is alive.# It cannot keep a Colab runtime alive after Google disconnects or reclaims it.import threadingwatchdog_stop = threading.Event()watchdog_state = {"api_failures": 0, "tunnel_failures": 0}def _wait_local_ollama(timeout=90):    deadline = time.time() + timeout    while time.time() < deadline:        try:            requests.get("http://127.0.0.1:11434/api/tags", timeout=3).raise_for_status()            return True        except Exception:            time.sleep(3)    return Falsedef _start_api_process():    global api_proc, api_log    api_log = open('/content/jassim-api.log', 'a')    api_proc = subprocess.Popen(        ['python', str(SERVER)], cwd=str(PROJECT/'backend'),        env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT    )def _start_tunnel_process():    global cf_proc, cf_log, tunnel_url    cf_log = open('/content/cloudflared.log', 'a')    cf_proc = subprocess.Popen(        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],        stdout=cf_log, stderr=subprocess.STDOUT, text=True    )    new_url = None    for _ in range(30):        time.sleep(2)        log_text = Path('/content/cloudflared.log').read_text(errors='ignore')        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)        if matches:            new_url = matches[-1]            break        if cf_proc.poll() is not None:            break    if new_url:        tunnel_url = new_url        print('JASSIM tunnel URL (update Android/dashboard if changed):', tunnel_url)    else:        print('Tunnel restart did not produce a URL; inspect /content/cloudflared.log')def _jassim_watchdog():    global ollama_proc, ollama_log, api_proc, cf_proc    while not watchdog_stop.wait(25):        try:            if ollama_proc.poll() is not None:                print('[watchdog] Ollama stopped; restarting...')                ollama_log = open('/content/ollama.log', 'a')                ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)                if not _wait_local_ollama():                    print('[watchdog] Ollama did not recover; check /content/ollama.log')                    continue            if api_proc.poll() is not None:                print('[watchdog] JASSIM API stopped; restarting...')                _start_api_process()                time.sleep(3)            try:                requests.get('http://127.0.0.1:8787/health', timeout=5).raise_for_status()                watchdog_state['api_failures'] = 0            except Exception:                watchdog_state['api_failures'] += 1                if watchdog_state['api_failures'] >= 3:                    print('[watchdog] API health failed repeatedly; restarting API...')                    if api_proc.poll() is None:                        api_proc.terminate()                        try: api_proc.wait(timeout=5)                        except subprocess.TimeoutExpired: api_proc.kill()                    _start_api_process()                    watchdog_state['api_failures'] = 0            if cf_proc.poll() is not None:                print('[watchdog] Cloudflare tunnel process stopped; restarting...')                _start_tunnel_process()            else:                try:                    requests.get(tunnel_url + '/health', timeout=8).raise_for_status()                    watchdog_state['tunnel_failures'] = 0                except Exception:                    watchdog_state['tunnel_failures'] += 1                    if watchdog_state['tunnel_failures'] >= 3:                        print('[watchdog] Public tunnel health failed repeatedly; replacing tunnel...')                        cf_proc.terminate()                        try: cf_proc.wait(timeout=5)                        except subprocess.TimeoutExpired: cf_proc.kill()                        _start_tunnel_process()                        watchdog_state['tunnel_failures'] = 0        except Exception as exc:            print('[watchdog] Monitor error:', type(exc).__name__)watchdog_thread = threading.Thread(target=_jassim_watchdog, daemon=True)watchdog_thread.start()print('JASSIM watchdog active. It restarts crashed services only while this Colab runtime is alive.')print("To stop the watchdog manually: watchdog_stop.set()")